# Vehicle Detection, Tracking and License Plate Recognition System

Pipeline: Video -> OpenCV -> YOLO11 (vehicles) -> ByteTrack (IDs) -> plate detector -> OCR -> voting -> `plates.txt`

**Runtime:** use a GPU (Runtime -> Change runtime type -> T4 GPU).

**Input:** upload your own video to Colab as `/content/traffic.mp4` (or change `VIDEO_PATH` in the config cell).

**Run order:** sections 0-6 once per session, then Stage B (slow), Stage C, and optionally Stage D.
After a Colab disconnect, upload the video again and run sections 0-6 before Stage B
(select the last cell of section 6 and use *Runtime -> Run before*).

**Privacy:** license plates can identify people. This notebook contains no real plate texts and no plate images.
Set `MASK_PLATES = True` in the config before rendering any video you plan to share.


# Part 1: Pipeline (project cells)

## 0. Setup

In [ ]:
!pip install -q ultralytics lap easyocr rapidfuzz "fast-plate-ocr[onnx]"

## 1. Imports

In [ ]:
import re
import pathlib
from collections import defaultdict

import cv2
import easyocr
import ultralytics
from ultralytics import YOLO
from huggingface_hub import hf_hub_download
from rapidfuzz.distance import Levenshtein
from fast_plate_ocr import LicensePlateRecognizer

## 2. Config

In [ ]:
VIDEO_PATH = "/content/traffic.mp4"
PLATE_OUTPUT_PATH = "output_plates.mp4"
FINAL_OUTPUT_PATH = "output_final.mp4"   # video with the final plate text above each vehicle
RESULTS_PATH = "plates.txt"
MASK_PLATES = False   # True: blur detected plates and hide plate text in the final video (use for any public demo)

# Vehicle detection + tracking
MODEL_PATH = "yolo11n.pt"
CONF_THRESHOLD = 0.3
VEHICLE_CLASSES = [2, 3, 5, 7]   # COCO: car, motorcycle, bus, truck
TRACK_BUFFER = 90
AGNOSTIC_NMS = True              # one box per object even if two classes were predicted

# Plate detection
PLATE_REPO = "Nuurrr66/license-plate-detector"
PLATE_FILE = "license-plate-finetune-v1n.pt"
PLATE_CONF = 0.25
PLATE_PAD = 0.0                  # extra margin around the plate box (fraction of its size); 0.12 was inconclusive

# OCR
OCR_ENGINE = "fast"     # "fast" = fast-plate-ocr (plate-specific), "easyocr" = general OCR
PLATE_PREFIX = "B"       # format rule for the plates in MY test video only (plates start with this letter); None = off. Not general.
ENHANCE_MODE = "none"    # none, upscale, clahe, sharpen, denoise, or combos like "clahe+sharpen" (none was as good as any other in my tests)

# OCR selection rules (values used in the evaluated run; not tuned on a held-out set)
OCR_TOP_K = 5        # how many of the best plate crops to read per vehicle
MIN_PLATE_H = 15     # ignore plates shorter than this many pixels
MIN_RATIO = 2.0      # plate width / height range
MAX_RATIO = 5.5
MIN_TEXT_LEN = 4     # ignore OCR reads shorter than this

## 3. Tracker config (ByteTrack with custom track_buffer)

In [ ]:
cfg = pathlib.Path(ultralytics.__file__).parent / "cfg/trackers/bytetrack.yaml"
custom_cfg = cfg.read_text().replace("track_buffer: 30", f"track_buffer: {TRACK_BUFFER}")
assert f"track_buffer: {TRACK_BUFFER}" in custom_cfg  # fail loudly if the replace did nothing
pathlib.Path("bytetrack_custom.yaml").write_text(custom_cfg)

## 4. Vehicle model
Re-run this cell **before every video run**: the tracker state lives inside the model object.

In [ ]:
model = YOLO(MODEL_PATH)

## 5. Plate model + OCR reader (once per session)

In [ ]:
plate_model = YOLO(hf_hub_download(repo_id=PLATE_REPO, filename=PLATE_FILE))
reader = easyocr.Reader(["en"], gpu=True)
fast_ocr = LicensePlateRecognizer("cct-s-v2-global-model")
print(plate_model.names)

{0: 'License_Plate'}


## 6. Helper functions

In [ ]:
def touches_border(px1, py1, px2, py2, crop_w, crop_h, margin=3):
    """True if the plate box touches an edge of the vehicle crop (likely a cut-off plate)."""
    return (px1 <= margin or py1 <= margin
            or px2 >= crop_w - margin or py2 >= crop_h - margin)


def normalize(text):
    """Uppercase and keep only letters and digits."""
    return re.sub(r"[^A-Z0-9]", "", text.upper())


def enhance(img, mode="none"):
    """Return an enhanced copy of a plate crop (BGR). Steps can be combined with '+'."""
    out = img
    for step in mode.split("+"):
        if step == "none":
            continue
        elif step == "upscale":
            out = cv2.resize(out, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
        elif step == "clahe":
            gray = cv2.cvtColor(out, cv2.COLOR_BGR2GRAY)
            gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4)).apply(gray)
            out = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        elif step == "sharpen":
            blur = cv2.GaussianBlur(out, (0, 0), 1.2)
            out = cv2.addWeighted(out, 1.5, blur, -0.5, 0)
        elif step == "denoise":
            out = cv2.bilateralFilter(out, 5, 40, 40)
        else:
            raise ValueError(f"Unknown enhancement step: {step}")
    return out


def ocr_read(img, mode=None):
    """Read one plate crop with EasyOCR. Returns (text, confidence); text is '' if nothing was read."""
    img = enhance(img, ENHANCE_MODE if mode is None else mode)
    results = reader.readtext(img)
    if not results:
        return "", 0.0
    text = normalize("".join(t for _, t, _ in results))
    conf = sum(float(c) for _, _, c in results) / len(results)
    return text, conf


def is_readable(it):
    """True if a stored plate crop is worth sending to OCR."""
    ratio = it["w"] / it["h"]
    return (not it["partial"]
            and it["h"] >= MIN_PLATE_H
            and MIN_RATIO <= ratio <= MAX_RATIO)


def choose_plate(reads):
    """reads: list of (text, conf). Returns (text, conf, n_valid_reads) or None.
    Picks the read that agrees most with all the other reads."""
    reads = [r for r in reads if len(r[0]) >= MIN_TEXT_LEN]
    if not reads:
        return None

    def total_distance(candidate):
        return sum(Levenshtein.distance(candidate[0], other[0]) for other in reads)

    best = min(reads, key=lambda r: (total_distance(r), -r[1]))
    return best[0], best[1], len(reads)


def ocr_fast(img, mode=None):
    """Read one plate crop with the plate-specific model. Returns (text, 0.0).
    The confidence is NOT used yet (always 0.0), so the avg_conf column in plates.txt is not meaningful."""
    img = enhance(img, ENHANCE_MODE if mode is None else mode)
    try:
        out = fast_ocr.run(img)
    except Exception:
        # API detail not verified: fall back to passing a file path
        cv2.imwrite("/tmp/_plate.png", img)
        out = fast_ocr.run("/tmp/_plate.png")
    first = out[0] if isinstance(out, (list, tuple)) else out
    text = getattr(first, "plate", first)   # newer versions return objects with .plate
    return normalize(str(text)), 0.0


def read_plate(img):
    """Read one plate crop with the engine selected in the config."""
    return ocr_fast(img) if OCR_ENGINE == "fast" else ocr_read(img)


def fix_prefix(text, prefix="B"):
    """Format rule for this camera: plates start with `prefix`. Dataset-specific."""
    i = text.find(prefix)
    if i == 1:            # one junk character before the prefix: drop it
        return text[1:]
    if i != 0:            # prefix missing (or buried deeper): add it
        return prefix + text
    return text

## 7. Stage B: process the video (track vehicles, detect plates, store plate crops)
Slow cell. Run it once per video / setting change.

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise IOError(f"Cannot open video: {VIDEO_PATH}")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(PLATE_OUTPUT_PATH, fourcc, fps, (width, height))

plate_tracks = defaultdict(list)   # track_id -> list of stored plate crops
seen_ids = set()
frame_boxes = defaultdict(list)   # frame_idx -> vehicles + plate boxes (used to render the final video)
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    clean = frame.copy()           # never drawn on: used for all crops

    result = model.track(
        frame,
        persist=True,
        tracker="bytetrack_custom.yaml",
        conf=CONF_THRESHOLD,
        classes=VEHICLE_CLASSES,
        agnostic_nms=AGNOSTIC_NMS,
        verbose=False,
    )[0]

    # Pass 1: collect vehicle crops
    vehicles = []
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        x1, y1 = max(x1, 0), max(y1, 0)
        x2, y2 = min(x2, width), min(y2, height)
        crop = clean[y1:y2, x1:x2]
        if crop.size == 0:
            continue
        track_id = int(box.id[0]) if box.id is not None else None
        if track_id is not None:
            seen_ids.add(track_id)
        vehicles.append({
            "id": track_id,
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "w": x2 - x1, "h": y2 - y1,
            "crop": crop,
        })

    # Detect plates in all vehicle crops of this frame (one batch)
    crops = [v["crop"] for v in vehicles]
    plate_results = (
        plate_model.predict(crops, conf=PLATE_CONF, verbose=False) if crops else []
    )

    # Pass 2: draw and store
    for v, pres in zip(vehicles, plate_results):
        cv2.rectangle(frame, (v["x1"], v["y1"]), (v["x2"], v["y2"]), (0, 0, 255), 1)
        if v["id"] is not None:
            cv2.putText(frame, f"ID {v['id']}", (v["x1"], max(v["y1"] - 6, 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

        plate_boxes = []
        for pbox in pres.boxes:
            px1, py1, px2, py2 = map(int, pbox.xyxy[0])
            fx1, fy1 = px1 + v["x1"], py1 + v["y1"]
            fx2, fy2 = px2 + v["x1"], py2 + v["y1"]
            plate_conf = float(pbox.conf[0])
            partial = touches_border(px1, py1, px2, py2, v["w"], v["h"])
            plate_boxes.append((fx1, fy1, fx2, fy2))

            color = (0, 165, 255) if partial else (0, 255, 0)  # BGR: orange / green
            cv2.rectangle(frame, (fx1, fy1), (fx2, fy2), color, 2)
            cv2.putText(frame, f"plate {plate_conf:.2f}", (fx1, max(fy1 - 6, 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

            # Stored crop: optionally padded; w/h stay the detector's tight size (used by the filters)
            pad_x = int((fx2 - fx1) * PLATE_PAD)
            pad_y = int((fy2 - fy1) * PLATE_PAD)
            qx1, qy1 = max(fx1 - pad_x, 0), max(fy1 - pad_y, 0)
            qx2, qy2 = min(fx2 + pad_x, width), min(fy2 + pad_y, height)
            plate_img = clean[qy1:qy2, qx1:qx2]

            if v["id"] is not None and plate_img.size > 0:
                plate_tracks[v["id"]].append({
                    "frame": frame_idx,
                    "img": plate_img.copy(),
                    "w": fx2 - fx1,
                    "h": fy2 - fy1,
                    "conf": plate_conf,
                    "partial": partial,
                })

        frame_boxes[frame_idx].append({
            "id": v["id"],
            "box": (v["x1"], v["y1"], v["x2"], v["y2"]),
            "plates": plate_boxes,
        })

    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()
print(f"Processed {frame_idx} frames | vehicles tracked: {len(seen_ids)} | "
      f"vehicles with a plate: {len(plate_tracks)}")

Processed 1285 frames | vehicles tracked: 32 | vehicles with a plate: 29


## 8. Stage C: read the best crops of each vehicle, vote, write `plates.txt`
Fast cell. Re-run it freely while tuning the OCR rules (no need to re-run Stage B).

In [ ]:
lines = ["id,plate,valid_reads,avg_conf"]
final_plates = {}   # track_id -> plate text, or "UNREADABLE" / "NO_PLATE_DETECTED"


for tid in sorted(seen_ids):
    if tid not in plate_tracks:
        lines.append(f"{tid},NO_PLATE_DETECTED,0,0.00")
        final_plates[tid] = "NO_PLATE_DETECTED"
        continue

    candidates = [it for it in plate_tracks[tid] if is_readable(it)]
    candidates.sort(key=lambda it: it["w"] * it["h"], reverse=True)
    top = candidates[:OCR_TOP_K]

    reads = [read_plate(it["img"]) for it in top]
    if PLATE_PREFIX:
        reads = [(fix_prefix(t, PLATE_PREFIX), c) for t, c in reads]
    choice = choose_plate(reads)

    if choice is None:
        lines.append(f"{tid},UNREADABLE,0,0.00")
        final_plates[tid] = "UNREADABLE"
    else:
        text, conf, n_reads = choice
        lines.append(f"{tid},{text},{n_reads},{conf:.2f}")
        final_plates[tid] = text

with open(RESULTS_PATH, "w") as f:
    f.write("\n".join(lines) + "\n")

print("\n".join(lines))
print(f"\nSaved to {RESULTS_PATH}")

## 9. Stage D: render the final video (plate text above each vehicle box)
Needs Stage B (`frame_boxes`) and Stage C (`final_plates`). No model is run here, so it is fast and uses exactly the same IDs.

In [ ]:
def pixelate(img, x1, y1, x2, y2, block=12):
    """Pixelate a rectangle in place (used to hide a detected plate)."""
    h_img, w_img = img.shape[:2]
    x1, y1, x2, y2 = max(x1, 0), max(y1, 0), min(x2, w_img), min(y2, h_img)
    w, h = x2 - x1, y2 - y1
    if w < 2 or h < 2:
        return
    roi = img[y1:y2, x1:x2]
    small = cv2.resize(roi, (max(1, w // block), max(1, h // block)), interpolation=cv2.INTER_LINEAR)
    img[y1:y2, x1:x2] = cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)


def draw_label(img, text, x, y, color, scale=0.8, thickness=2):
    """Draw text on a filled background just above (x, y); inside the box if there is no room above."""
    font = cv2.FONT_HERSHEY_SIMPLEX
    (tw, th), base = cv2.getTextSize(text, font, scale, thickness)
    h = th + base + 6
    top = y - h
    if top < 0:                                   # box touches the top edge of the frame
        top = y
    x = max(0, min(x, img.shape[1] - tw - 8))     # keep the label inside the frame
    cv2.rectangle(img, (x, top), (x + tw + 8, top + h), color, -1)
    cv2.putText(img, text, (x + 4, top + th + 3), font, scale, (255, 255, 255), thickness)


COLOR_READ = (0, 150, 0)
COLOR_UNREADABLE = (0, 140, 255)
COLOR_NO_PLATE = (90, 90, 90)
COLOR_PLATE_BOX = (0, 255, 255)

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise IOError(f"Cannot open video: {VIDEO_PATH}")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
label_scale = min(1.0, max(0.5, width / 2000))   # keep the label size reasonable for any resolution

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(FINAL_OUTPUT_PATH, fourcc, fps, (width, height))

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    for v in frame_boxes.get(frame_idx, []):
        x1, y1, x2, y2 = v["box"]
        tid = v["id"]
        id_text = "?" if tid is None else str(tid)
        status = final_plates.get(tid, "NO_PLATE_DETECTED")

        if status == "UNREADABLE":
            color, label = COLOR_UNREADABLE, f"ID {id_text} | ?"
        elif status == "NO_PLATE_DETECTED":
            color, label = COLOR_NO_PLATE, f"ID {id_text}"
        else:
            shown = status[0] + "*" * (len(status) - 1) if MASK_PLATES else status
            color, label = COLOR_READ, f"ID {id_text} | {shown}"

        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        draw_label(frame, label, x1, y1, color, scale=label_scale)

        for (fx1, fy1, fx2, fy2) in v["plates"]:
            if MASK_PLATES:
                pixelate(frame, fx1, fy1, fx2, fy2)
            cv2.rectangle(frame, (fx1, fy1), (fx2, fy2), COLOR_PLATE_BOX, 2)

    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()
print(f"Saved {frame_idx} frames to {FINAL_OUTPUT_PATH}")

Saved 1285 frames to output_final.mp4


---
# Part 2: Experiments and evaluation (NOT part of the pipeline)
Cells below measure and justify the choices made in Part 1. They are not needed to produce `plates.txt`.
Evaluation needs a local `ground_truth.csv` (hand-written plate texts, never committed); without it those cells are skipped.


### E2. Duplicate boxes on the same object (found the car/truck problem -> agnostic_nms)

In [ ]:
def iou(a, b):
    """Intersection over Union of two boxes in (x1, y1, x2, y2) format."""
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (area_a + area_b - inter + 1e-9)


def count_duplicates(**track_kwargs):
    m = YOLO(MODEL_PATH)  # fresh model = fresh tracker state
    dup_frames = 0
    class_pairs = {}
    cap = cv2.VideoCapture(VIDEO_PATH)
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        r = m.track(frame, persist=True, tracker="bytetrack_custom.yaml",
                    conf=CONF_THRESHOLD, classes=VEHICLE_CLASSES,
                    verbose=False, **track_kwargs)[0]
        boxes = r.boxes.xyxy.tolist()
        names = [m.names[int(c)] for c in r.boxes.cls.tolist()]
        found = False
        for i in range(len(boxes)):
            for j in range(i + 1, len(boxes)):
                if iou(boxes[i], boxes[j]) > 0.5:
                    found = True
                    key = tuple(sorted((names[i], names[j])))
                    class_pairs[key] = class_pairs.get(key, 0) + 1
        dup_frames += found
    cap.release()
    return dup_frames, class_pairs


print("default      :", count_duplicates())
print("agnostic_nms :", count_duplicates(agnostic_nms=True))

default      : (183, {('car', 'truck'): 186, ('bus', 'car'): 6, ('car', 'car'): 6, ('bus', 'truck'): 6})
agnostic_nms : (6, {('car', 'car'): 6})


### E3. Number of IDs for a given tracking setting

In [ ]:
def count_ids(**track_kwargs):
    m = YOLO(MODEL_PATH)  # fresh model = fresh tracker state
    ids = set()
    cap = cv2.VideoCapture(VIDEO_PATH)
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        r = m.track(frame, persist=True, tracker="bytetrack_custom.yaml",
                    conf=CONF_THRESHOLD, classes=VEHICLE_CLASSES,
                    verbose=False, **track_kwargs)[0]
        if r.boxes.id is not None:
            ids.update(int(i) for i in r.boxes.id.tolist())
    cap.release()
    return ids


print("default      :", len(count_ids()))
print("agnostic_nms :", len(count_ids(agnostic_nms=True)))

default      : 90
agnostic_nms : 32


### E4. Helper: look at plate crops (to read them by eye and write ground truth)
Definition only. Run it locally; do not commit notebooks with these images in their outputs.

In [ ]:
from google.colab.patches import cv2_imshow


def show_plates(ids, n=3, scale=4):
    """Show the n largest readable plate crops of each ID, enlarged."""
    for tid in ids:
        cands = [it for it in plate_tracks.get(tid, []) if is_readable(it)]
        cands.sort(key=lambda it: it["w"] * it["h"], reverse=True)
        print(f"--- ID {tid} ({len(cands)} readable crops) ---")
        for it in cands[:n]:
            cv2_imshow(cv2.resize(it["img"], None, fx=scale, fy=scale,
                                  interpolation=cv2.INTER_CUBIC))



### E5. Evaluation against hand-written plate texts (ground truth)
Ground truth lives in a local file `ground_truth.csv` (columns: `track_id,plate_text`), **not** in this repository.
A `?` marks a character that could not be read by eye and is treated as a wildcard when scoring.
Track IDs belong to ONE run: if tracking settings change, IDs change, so check with `show_plates` before trusting a score.


In [ ]:
import csv
import os

GT_PATH = "ground_truth.csv"   # local file, NOT committed (listed in .gitignore)


def load_ground_truth(path=GT_PATH):
    """Read {track_id: plate_text} from a CSV with columns: track_id, plate_text."""
    if not os.path.exists(path):
        return {}
    with open(path, newline="") as f:
        return {int(r["track_id"]): r["plate_text"] for r in csv.DictReader(f)}


GROUND_TRUTH = load_ground_truth()


def normalize_gt(text):
    """Like normalize(), but keeps '?' (uncertain character)."""
    return re.sub(r"[^A-Z0-9?]", "", text.upper())


def wild_distance(pred, gt):
    """Levenshtein distance where a '?' in gt matches any single character for free."""
    prev = list(range(len(gt) + 1))
    for i, cp in enumerate(pred, 1):
        cur = [i]
        for j, cg in enumerate(gt, 1):
            sub = 0 if (cp == cg or cg == "?") else 1
            cur.append(min(prev[j] + 1, cur[-1] + 1, prev[j - 1] + sub))
        prev = cur
    return prev[-1]


def read_results(path):
    """Parse plates.txt into {track_id: plate_text}."""
    out = {}
    with open(path) as f:
        next(f)  # skip header line
        for line in f:
            tid, plate, *_ = line.strip().split(",")
            out[int(tid)] = plate
    return out


def evaluate_results(preds, truths, post=None, verbose=False):
    """Final-text evaluation: one voted text per ID."""
    cers, exact = [], 0
    for tid, gt_raw in truths.items():
        gt = normalize_gt(gt_raw)
        pred = preds.get(tid, "")
        if pred in ("UNREADABLE", "NO_PLATE_DETECTED"):
            pred = ""
        if post is not None:
            pred = post(pred)
        d = wild_distance(pred, gt)
        cers.append(d / len(gt))
        exact += (d == 0)
        if verbose:
            print(f"ID {tid:3d}: truth={gt:9s} pred={pred:9s} CER={d / len(gt):.2f}")
    print(f"{len(cers)} plates | mean CER = {sum(cers) / len(cers):.2f} | exact matches = {exact}/{len(cers)}")


def per_read_eval(tracks, truths, ocr_fn=ocr_read, n_reads=15, post=None):
    """Read the n_reads best crops of each labeled ID separately and compare the mean error
    of single reads with the error of the voted result. IDs with no readable crop are skipped."""
    read_cers, voted_cers, skipped = [], [], []
    for tid, gt_raw in truths.items():
        gt = normalize_gt(gt_raw)
        cands = [it for it in tracks.get(tid, []) if is_readable(it)]
        cands.sort(key=lambda it: it["w"] * it["h"], reverse=True)
        reads = [ocr_fn(it["img"]) for it in cands[:n_reads]]
        if not reads:
            skipped.append(tid)
            continue
        if post is not None:
            reads = [(post(t), c) for t, c in reads]

        read_cers += [wild_distance(t, gt) / len(gt) for t, _ in reads]
        voted = choose_plate(reads[:OCR_TOP_K])
        voted_cers.append(wild_distance(voted[0], gt) / len(gt) if voted else 1.0)

    if read_cers:
        print(f"{len(voted_cers)} plates, {len(read_cers)} reads | "
              f"mean single-read CER = {sum(read_cers) / len(read_cers):.2f} | "
              f"mean voted CER = {sum(voted_cers) / len(voted_cers):.2f}"
              + (f" | skipped (no readable crops): {skipped}" if skipped else ""))


### E6. Run the evaluation (skipped automatically if `ground_truth.csv` is missing)
Reference results from my run (20 hand-labeled plates, one video):
- Final text (prefix rule already applied in Stage C): exact match 7/20, mean CER 0.12.
- OCR engine comparison, voted CER: EasyOCR 0.39 -> fast-plate-ocr 0.23 -> fast-plate-ocr + prefix rule 0.12.
- Image enhancement (upscale, CLAHE, sharpen, denoise, CLAHE+sharpen) gave no clear gain: differences of 0.02 CER or less, within noise on 20 plates. `none` is kept.
- Limitation: the same 20 plates were used to choose the engine, the prefix rule and the enhancement mode, so there is no held-out test set.


In [ ]:
if not GROUND_TRUTH:
    print("ground_truth.csv not found: skipping evaluation.")
else:
    print("--- final text (prefix rule is already applied in Stage C)")
    evaluate_results(read_results(RESULTS_PATH), GROUND_TRUTH)

    print("--- OCR engine comparison on the same crops")
    for name, fn, post in [("EasyOCR, raw", ocr_read, None),
                           ("fast-plate-ocr, raw", ocr_fast, None),
                           ("fast-plate-ocr + prefix rule", ocr_fast, fix_prefix)]:
        print(f"[{name}]")
        per_read_eval(plate_tracks, GROUND_TRUTH, ocr_fn=fn, post=post)
